## Pathing, parameters and setup

In [ ]:
import h5py
import pandas as pd
import scipy

import numpy as np
from pathlib import Path
from tqdm import tqdm

projectRoot = Path().resolve().parent


## USER SETTINGS

`dataDriveRoot` is the root of the drive/mount holding the MATLAB reconstruction
output (i.e. the same `driveName` used throughout the `imageReconstruction` MATLAB
repo). Edit this to match your machine — `pathlib.Path` handles the rest
cross-platform, so no OS-specific branching is needed here.

In [ ]:
dataDriveRoot = Path('/Volumes/YourDriveName')  # <-- edit: external drive mount point, or a local/mapped folder


## Surface or volumetric parcellation?

In [ ]:
surfaceParcellation = 1  # set to 0 for volumetric (parcelHb) and 1 for surface-based (surfParcelHb)


## Find parcel-wise Hb files

`surfParcelHb` / `parcelHb` are the Tier 2 output folders produced by
`parcelCollationSurfaceHaND.m` / `parcelCollationVolumetricHaND.m` in the
`imageReconstruction` MATLAB repo (see `saveFiles.m` for exactly how these are named
and structured).

In [ ]:
if surfaceParcellation == 1:
    rootDir = dataDriveRoot / 'dot' / 'derivatives' / 'surfParcelHb'
    outputFilenameMetadata = project_root / 'dataRaw' / 'indigoSurfParcelMetadataStandard.csv'
    outputFilenameTimeseries = project_root / 'dataRaw' / 'indigoSurfParcelTrialsVlenStandard.h5'
else:
    rootDir = dataDriveRoot / 'dot' / 'derivatives' / 'parcelHb'
    outputFilenameMetadata = project_root / 'dataRaw' / 'indigoParcelMetadataStandard.csv'
    outputFilenameTimeseries = project_root / 'dataRaw' / 'indigoParcelTrialsVlenStandard.h5'

matFiles = [f for f in rootDir.rglob('*.mat') if 'hand' in f.name.lower()]
matFiles = [f for f in matFiles if not f.name.startswith('.')]  # omit AppleDouble files, .DS_Store etc. (harmless no-op on Windows)


## Data checking

Reference cell: shows the nested structure of a saved `parcelData`/`surfParcelData`
`.mat` file once loaded via `scipy.io.loadmat` (works because these files are saved
in MATLAB's default, pre-v7.3 format — not HDF5 — by `saveFiles.m`; if that ever
changes to `-v7.3`, this would need `h5py` instead).

The nesting below mirrors `parcelData`'s field order exactly as set in
`parcelCollationSurfaceHaND.m`: `blockData`, `blockAvgData`, `trialNumbers`,
`parcelNumbers`, `capName`.

IMPORTANT: the `[iChrom][iParcel][iTrial]` indexing order below only works because
of a `permute(..., [1 3 2])` applied to `blockData` right before saving, on the
MATLAB side (see `parcelCollationSurfaceHaND.m`). If that permute is ever removed or
changed, this indexing breaks silently — it's not visible anywhere on the Python side.

In [ ]:
exampleFile = scipy.io.loadmat(matFiles[0])
exampleFile = exampleFile['parcelData']
# exampleFile[0][0][0]              # block data
# exampleFile[0][0][0][1][0][12]    # block time series for all trials in the 13th parcel, for HbR (2nd chromophore)
# exampleFile[0][0][0][1][0][12][5] # block time series for the 6th trial in the 13th parcel, for HbR
# exampleFile[0][0][1][0][0]        # block average data - not used here
# exampleFile[0][0][2]               # trial numbers
# exampleFile[0][0][2][16][0]        # 17th trial number
# exampleFile[0][0][3][1]            # parcel numbers (HbR)
# exampleFile[0][0][3][1][0][5][0]   # 6th parcel number for HbR
# exampleFile[0][0][4][0]            # cap name


## Create dataframe and time series storage files

Flattens every valid (infant × age × chromophore × parcel × trial) time series into
a single variable-length HDF5 dataset, with a matching metadata CSV (one row per
time series, referencing its index into the HDF5 dataset).

In [ ]:
metadataRows = []

with h5py.File(outputFilenameTimeseries, 'w') as h5f:
    # Variable-length (vlen) dataset
    dt = h5py.vlen_dtype(np.dtype('float32'))
    tsDataset = h5f.create_dataset('timeSeries', shape=(0,), maxshape=(None,), dtype=dt)

    idx = 0
    for matFile in tqdm(matFiles, desc="Files"):

        dataHb = scipy.io.loadmat(matFile)
        dataHb = dataHb['parcelData']

        # Fixed-position filename parsing, e.g. 'sub-002b_ses-12_task-hand_run-02_preproc_parcelHb.mat'.
        # Only the first 3 digits of the infant ID are taken (matches the convention used
        # in getInfantCap.m on the MATLAB side, where the trailing check-letter is
        # likewise dropped) - relies on this exact BIDS filename structure holding.
        infantID = matFile.name[4:7]
        age = matFile.name[13:15]
        run = matFile.name[30:32]
        capName = dataHb[0][0][4][0]

        numChroms = dataHb[0][0][3].shape[0]
        numTrials = dataHb[0][0][2].shape[0]

        for iChrom in range(0, numChroms):

            numParcels = dataHb[0][0][3][iChrom][0].shape[0]

            numChrom = iChrom + 1

            for iParcel in range(0, numParcels):

                numParcel = dataHb[0][0][3][iChrom][0][iParcel][0]

                for iTrial in range(0, numTrials):

                    numTrial = dataHb[0][0][2][iTrial][0]

                    timeSeries = dataHb[0][0][0][iChrom][0][iParcel][iTrial]

                    timeSeries = timeSeries.astype('float32')  # convert to float32 to save space

                    # skip if time series is NaNs
                    if np.all(np.isnan(timeSeries)):
                        continue

                    # Resize dataset to append
                    tsDataset.resize((idx + 1,))
                    tsDataset[idx] = timeSeries  # save variable-length array

                    metadataRows.append([infantID, age, run, numChrom, numParcel, numTrial, capName, idx])
                    idx += 1

                    # Periodic flush so partial progress is recoverable if this is interrupted
                    if idx % 10 == 0:
                        h5f.flush()

    print('Saving metadata as .csv file')
    columns = ['Infant', 'Age', 'Run', 'Chrom', 'Parcel', 'Trial', 'Cap name', 'tsIDx']
    pd.DataFrame(metadataRows, columns=columns).to_csv(outputFilenameMetadata, index=False)

    h5f.flush()


## Print the first few rows of the dataframe...

In [ ]:
dfMetadata = pd.read_csv(outputFilenameMetadata)
dfMetadata.head()


## ...and the time series

In [ ]:
with h5py.File(outputFilenameTimeseries, 'r') as h5f:
    tsDataset = h5f['timeSeries']
    print("\nFirst few time series:")
    for i in range(5):
        ts = tsDataset[i][...]  # load the full array
        print(f"\nTime series {i} (length {len(ts)}):")
        print(ts[:10])  # show first 10 timepoints
